In [1]:
import os
from pathlib import Path
import sys

# Automatically find repo root by looking for .git
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

# FIX: fail loudly instead of silently falling back to cwd — otherwise every
# src.* import below fails with a confusing ModuleNotFoundError if this
# notebook is ever opened from outside the repo.
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Could not find repo root from {Path.cwd()}. "
        "Open this notebook from inside the project directory."
    )

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Change the working directory to the repo root
os.chdir(ROOT)

In [2]:
import pandas as pd

In [3]:
from src.utils.data_loaders.read_settings_json import read_settings_json

args = read_settings_json()
args

{'Config': {'debug_mode': 'False', 'TEMP_CACHE': 'data/temp_cache'},
 'TrainingInput': {'CHART_OF_ACCOUNTS': 'data/training_input/chart_of_accounts.xlsx',
  'ENROLLEES': 'data/training_input/enrollees_pseudonymized.xlsx',
  'REVENUES': 'data/training_input/revenues_pseudonymized.xlsx'},
 'Training': {'MODEL_PARAMETERS': 'src/modules/machine_learning/parameters.json',
  'RESULTS_ROOT': 'data/training_results',
  'LOGS': 'data/training_logs',
  'DEPLOYED_MODELS': 'data/training_results/deployed_models',
  'observation_end': '2026/05/09',
  'target_feature': 'dtp_bracket',
  'test_size': '0.30'}}

In [4]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

from src.modules.machine_learning.utils.inference.inference_pipeline import (
    find_deployed_model,
    load_inference_pipeline,
    run_batch_inference,
)

MODEL_DIR = args["Training"]["DEPLOYED_MODELS"]
SHOW_PROBA_COLS = True  # set True to include prob_* columns in display

In [5]:
# Locate and inspect the deployed artifact.
# Raises ValueError with upgrade instructions if artifact is pre-InferencePipeline.
artifact_path = find_deployed_model(MODEL_DIR)
print("Artifact:", artifact_path)

try:
    pipeline = load_inference_pipeline(MODEL_DIR)
    print(pipeline)
except ValueError as e:
    msg = str(e)
    print("[WARN] Could not load InferencePipeline:")
    print(" ", msg)
    print()
    print("Action required: Re-run Step 5 (Model Finalization) in the app to regenerate the artifact.")
    pipeline = None


INFO src.modules.machine_learning.utils.inference.inference_pipeline Loading InferencePipeline from data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


Artifact: data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


INFO src.modules.machine_learning.utils.inference.inference_pipeline Loaded InferencePipeline(
  model_key         = 'two_stage_xgb_ada'
  lda_transformer   = None
  time_points       = 9 points
  classes           = [np.str_('30_days'), np.str_('60_days'), np.str_('90_days'), np.str_('on_time')]
  feature_metadata  = ['plan_risk_map']
)


InferencePipeline(
  model_key         = 'two_stage_xgb_ada'
  lda_transformer   = None
  time_points       = 9 points
  classes           = [np.str_('30_days'), np.str_('60_days'), np.str_('90_days'), np.str_('on_time')]
  feature_metadata  = ['plan_risk_map']
)


In [6]:
# FIX: python-calamine isn't part of the standard pandas install set —
# fall back to openpyxl so the notebook doesn't hard-fail on a missing
# optional dependency.
try:
    df_revenues = pd.read_excel(args['TrainingInput']['REVENUES'], engine='calamine')
except ImportError:
    df_revenues = pd.read_excel(args['TrainingInput']['REVENUES'])

In [7]:
# FIX: same calamine fallback as the revenues cell above.
try:
    df_enrollees = pd.read_excel(args['TrainingInput']['ENROLLEES'], engine='calamine')
except ImportError:
    df_enrollees = pd.read_excel(args['TrainingInput']['ENROLLEES'])

In [8]:
from src.modules.feature_engineering.credit_sales_machine_learning import CreditSalesProcessor
from datetime import datetime

# CreditSalesProcessor requires an object exposing `.observation_end` as a real
# datetime. Passing the raw settings dict (as before) means
# getattr(args, 'observation_end', ...) never finds the attribute and silently
# falls back to today's date -- desyncing the censor / dtp_bracket calculation
# from what step_3.py used to build the training data for this same artifact.
class _Cfg:
    observation_end = datetime.strptime(args["Training"]["observation_end"], "%Y/%m/%d")

# FIX: must mirror step_3.py's CreditSalesProcessor call exactly. The previous
# flag set here (drop_fully_paid_invoices, calculate_payment_amounts, and the
# missing drop_helper_columns/drop_demographic_columns/exclude_school_years/
# winsorise_dtp) produced a different column set and different numeric
# distributions than what the deployed scaler/classifier were actually fit
# on -- corrupting every prediction regardless of column-name validation.
# drop_fully_paid_invoices in particular dropped exactly the rows that have a
# real, observed dtp_bracket outcome, leaving only still-open invoices whose
# "actual_label" is a proxy "days late so far", not a true outcome.
#
# drop_missing_dtp=True drops invoices with no prior payment history (NaN
# dtp_1..4) -- every training-time call site uses True, and the Cox model
# raises "Input X contains NaN" if a NaN slips through.
#
# plan_risk_map is pulled from the loaded InferencePipeline's feature_metadata
# so plan_type_risk_score is scored against the training-time distribution
# instead of being recomputed from this small batch (Fit-Transform pattern --
# see model_analysis_callbacks.predict_on_new_data for the reference usage).
#
# add_description is intentionally NOT set here: InvoicePostProcessor.build()
# drops 'category_name' (via drop_demographic_columns) before it ever applies
# the description function, so add_description=True raises
# "KeyError: 'category_name'" whenever combined with drop_demographic_columns.
# Neither step_3.py (training) nor predict_on_new_data (the app's reference
# inference helper) pass add_description, so we don't either.
_plan_risk_map = pipeline.feature_metadata.get("plan_risk_map") if pipeline is not None else None
if pipeline is not None and _plan_risk_map is None:
    print("[WARN] Loaded pipeline has no plan_risk_map in feature_metadata -- "
          "plan_type_risk_score will be recomputed from this batch instead of "
          "the training distribution.")

cs_test = CreditSalesProcessor(
    df_revenues, df_enrollees, _Cfg(),
    drop_helper_columns=True,
    drop_demographic_columns=True,
    drop_plan_type_columns=False,
    drop_missing_dtp=True,
    drop_back_account_transactions=True,
    exclude_school_years=[2016, 2017, 2018],
    winsorise_dtp=True,
    plan_risk_map=_plan_risk_map,
)
df_cs_test = cs_test.show_data()
df_cs_test

Single due date records:   11151
Multiple due date records: 289
Excluded school years [2016, 2017, 2018]: removed 234 rows, 10215 remaining.
Dropped 3588 invoices with missing DTP values. Remaining: 6620


,credit_sale_amount,due_date,days_elapsed_until_fully_paid,dtp_1,dtp_2,dtp_3,dtp_4,dtp_avg,dtp_wavg,dtp_2_trend,...,dtp_bracket,censor,due_month,due_quarter,opening_balance_flag,payment_ratio,early_payer_flag,dtp_rolling_std,dtp_max,plan_type_risk_score
1659,2800.0,2022-02-07,-9,24,-5,-2,79,24.00,15.6,-0.491525,...,on_time,1,2,1,0,1.000000,0.0,38.910153,79,0
1752,2800.0,2022-04-04,-4,-9,24,-5,-2,2.00,2.4,0.589286,...,on_time,1,4,2,0,1.000000,1.0,14.944341,24,0
9777,4700.0,2026-02-06,-8,-2,-28,-14,-7,-12.75,-12.7,-0.412698,...,on_time,1,2,1,1,0.988662,1.0,11.295279,-2,0
10065,4933.0,2026-03-06,-2,-8,-2,-28,-14,-13.00,-10.8,0.214286,...,on_time,1,3,1,0,1.000000,1.0,11.135529,-2,0
571,2900.0,2019-12-06,8,33,193,152,284,165.50,129.9,6.400000,...,30_days,1,12,4,1,0.905309,0.0,104.142531,284,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9841,5800.0,2026-02-06,3,34,34,24,12,26.00,29.8,0.000000,...,30_days,1,2,1,1,0.972245,0.0,10.456258,34,0
9889,1500.0,2026-02-18,33,3,34,34,24,23.75,20.6,2.583333,...,60_days,1,2,1,1,0.992873,0.0,14.614491,34,0
9920,400.0,2026-03-05,6,33,3,34,34,26.00,24.3,-2.000000,...,30_days,1,3,1,1,0.990990,0.0,15.340578,34,0
10055,5800.0,2026-03-06,5,6,33,3,34,19.00,16.3,27.000000,...,30_days,1,3,1,1,0.964462,0.0,16.792856,34,0


In [9]:
# Mirror step_5's training-time row filter: the classifier was fit only on
# df_credit_sales[df_credit_sales['censor'] == 1] -- invoices with a known,
# observed payment outcome. censor == 0 rows are still-open invoices whose
# dtp_bracket is a proxy "days late so far" value, not a true historical
# outcome, so scoring them puts the model out-of-distribution and makes any
# comparison against actual_label meaningless.
df_cs_infer = df_cs_test[df_cs_test["censor"] == 1].copy()
actual_labels = df_cs_infer["dtp_bracket"].copy()

EXCLUDE = {"dtp_bracket", "days_elapsed_until_fully_paid", "censor", "date_fully_paid",
           "due_date", "school_year", "student_id_pseudonimized", "category_name",
           "description"}
X_infer = df_cs_infer.drop(columns=[c for c in EXCLUDE if c in df_cs_infer.columns])

print(f"Inference rows (censor == 1): {len(X_infer)} of {len(df_cs_test)} total")
X_infer

Inference rows (censor == 1): 6551 of 6620 total


,credit_sale_amount,dtp_1,dtp_2,dtp_3,dtp_4,dtp_avg,dtp_wavg,dtp_2_trend,dtp_3_trend,days_since_last_payment,...,plan_type_Plan - E,plan_type_nan,due_month,due_quarter,opening_balance_flag,payment_ratio,early_payer_flag,dtp_rolling_std,dtp_max,plan_type_risk_score
1659,2800.0,24,-5,-2,79,24.00,15.6,-0.491525,-0.412698,35,...,0.0,0.0,2,1,0,1.000000,0.0,38.910153,79,0
1752,2800.0,-9,24,-5,-2,2.00,2.4,0.589286,0.034783,65,...,0.0,0.0,4,2,0,1.000000,1.0,14.944341,24,0
9777,4700.0,-2,-28,-14,-7,-12.75,-12.7,-0.412698,-0.190476,63,...,0.0,0.0,2,1,1,0.988662,1.0,11.295279,-2,0
10065,4933.0,-8,-2,-28,-14,-13.00,-10.8,0.214286,-0.219780,21,...,0.0,0.0,3,1,0,1.000000,1.0,11.135529,-2,0
571,2900.0,33,193,152,284,165.50,129.9,6.400000,0.619792,38,...,0.0,0.0,12,4,1,0.905309,0.0,104.142531,284,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9841,5800.0,34,34,24,12,26.00,29.8,0.000000,-0.158730,29,...,0.0,0.0,2,1,1,0.972245,0.0,10.456258,34,0
9889,1500.0,3,34,34,24,23.75,20.6,2.583333,0.413333,9,...,0.0,0.0,2,1,1,0.992873,0.0,14.614491,34,0
9920,400.0,33,3,34,34,26.00,24.3,-2.000000,0.037037,24,...,0.0,0.0,3,1,1,0.990990,0.0,15.340578,34,0
10055,5800.0,6,33,3,34,19.00,16.3,27.000000,-0.187500,25,...,0.0,0.0,3,1,1,0.964462,0.0,16.792856,34,0


In [10]:
if pipeline is not None:
    print("Scaler expects these columns:")
    print(list(pipeline.scaler.feature_names_in_))
    print("\nNotebook is sending these columns:")
    print(list(X_infer.columns))
    print("\nMissing from notebook input (will cause silent NaN or wrong predictions):")
    print([c for c in pipeline.scaler.feature_names_in_ if c not in X_infer.columns])
    print("\nExtra in notebook input (will be silently dropped):")
    print([c for c in X_infer.columns if c not in list(pipeline.scaler.feature_names_in_)])

Scaler expects these columns:
['credit_sale_amount', 'dtp_1', 'dtp_2', 'dtp_3', 'dtp_4', 'dtp_avg', 'dtp_wavg', 'dtp_2_trend', 'dtp_3_trend', 'days_since_last_payment', 'amount_due_cumsum', 'amount_paid_cumsum', 'opening_balance', 'plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C', 'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan', 'due_month', 'due_quarter', 'opening_balance_flag', 'payment_ratio', 'early_payer_flag', 'dtp_rolling_std', 'dtp_max', 'plan_type_risk_score']

Notebook is sending these columns:
['credit_sale_amount', 'dtp_1', 'dtp_2', 'dtp_3', 'dtp_4', 'dtp_avg', 'dtp_wavg', 'dtp_2_trend', 'dtp_3_trend', 'days_since_last_payment', 'amount_due_cumsum', 'amount_paid_cumsum', 'opening_balance', 'plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C', 'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan', 'due_month', 'due_quarter', 'opening_balance_flag', 'payment_ratio', 'early_payer_flag', 'dtp_rolling_std', 'dtp_max', 'plan_type_risk_

In [11]:
if pipeline is not None:
    df_preds = run_batch_inference(
        input_source=X_infer,
        model_dir=MODEL_DIR,
        batch_size=1024,
        return_proba=True,
    )

    df_preds.insert(
        df_preds.columns.get_loc("predicted_label") + 1,
        "actual_label",
        actual_labels.reindex(df_preds.index),
    )

    prob_cols = [c for c in df_preds.columns if c.startswith("prob_")]
    if SHOW_PROBA_COLS and not prob_cols:
        print("[WARN] SHOW_PROBA_COLS=True but no prob_* columns are present "
              "(return_proba=False was passed to run_batch_inference).")
    display_cols = [
        c for c in df_preds.columns
        if c not in prob_cols or SHOW_PROBA_COLS
    ]
    display(df_preds[display_cols].head(10))
else:
    df_preds = None
    print("Skipping inference -- no valid InferencePipeline loaded.")

INFO src.modules.machine_learning.utils.inference.inference_pipeline Loading InferencePipeline from data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl
INFO src.modules.machine_learning.utils.inference.inference_pipeline Loaded InferencePipeline(
  model_key         = 'two_stage_xgb_ada'
  lda_transformer   = None
  time_points       = 9 points
  classes           = [np.str_('30_days'), np.str_('60_days'), np.str_('90_days'), np.str_('on_time')]
  feature_metadata  = ['plan_risk_map']
)
INFO src.modules.machine_learning.utils.inference.inference_pipeline Running batch inference on 6551 rows in chunks of 1024 (model=two_stage_xgb_ada)
INFO src.modules.machine_learning.utils.inference.inference_pipeline   chunk 0–1023 done
INFO src.modules.machine_learning.utils.inference.inference_pipeline   chunk 1024–2047 done
INFO src.modules.machine_learning.utils.inference.inference_pipeline   chunk 2048–3071 done
INFO src.modules.machine_learning.utils.inference.inference_pipelin

,predicted_label,actual_label,prob_30_days,prob_60_days,prob_90_days,prob_on_time,model_key,artifact_path,run_timestamp
1659,30_days,on_time,0.967449,0.011389,0.010861,0.010300,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
1752,30_days,on_time,0.965036,0.013092,0.011327,0.010545,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
9777,30_days,on_time,0.501908,0.200098,0.170899,0.127096,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
10065,30_days,on_time,0.983712,0.006350,0.005256,0.004681,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
571,on_time,30_days,0.029089,0.274004,0.311869,0.385038,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
1902,on_time,90_days,0.113683,0.259058,0.310089,0.317170,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
3939,30_days,on_time,0.592541,0.123158,0.136342,0.147959,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
5904,90_days,90_days,0.026925,0.322673,0.343813,0.306589,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
8254,90_days,30_days,0.164549,0.269854,0.300371,0.265226,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00
3451,30_days,on_time,0.952567,0.015414,0.015650,0.016369,two_stage_xgb_ada,C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\d...,2026-06-16T10:29:40.735299+00:00


In [12]:
if df_preds is not None:
    print("Predicted label distribution:")
    display(df_preds["predicted_label"].value_counts().rename("count").to_frame())
    n = len(df_preds)
    mk = df_preds["model_key"].iloc[0]
    ts = df_preds["run_timestamp"].iloc[0]
    print()
    print("Total rows scored:", n)
    print("Model key        :", mk)
    print("Run timestamp    :", ts)


Predicted label distribution:


,count
predicted_label,
30_days,2580
60_days,1825
on_time,1188
90_days,958



Total rows scored: 6551
Model key        : two_stage_xgb_ada
Run timestamp    : 2026-06-16T10:29:40.735299+00:00


In [13]:
if df_preds is not None and "actual_label" in df_preds.columns:
    eval_df = df_preds.dropna(subset=["actual_label", "predicted_label"])

    prob_col_map = {
        cls: f"prob_{cls}"
        for cls in pipeline.label_encoder.classes_
        if f"prob_{cls}" in eval_df.columns
    }
    y_proba = eval_df[list(prob_col_map.values())].values if prob_col_map else None

    metrics = pipeline.evaluate_predictions(
        y_true=eval_df["actual_label"],
        y_pred=eval_df["predicted_label"],
        y_proba=y_proba,
    )

    print(f"Rows evaluated : {len(eval_df)}")
    print(f"Accuracy       : {metrics['accuracy']:.4f}")
    print(f"Macro F1       : {metrics['f1_macro']:.4f}")
    print(f"Macro Precision: {metrics['precision_macro']:.4f}")
    print(f"Macro Recall   : {metrics['recall_macro']:.4f}")
    if metrics["roc_auc_macro"] is not None:
        print(f"ROC AUC (macro): {metrics['roc_auc_macro']:.4f}")

Rows evaluated : 6551
Accuracy       : 0.1073
Macro F1       : 0.1285
Macro Precision: 0.1212
Macro Recall   : 0.1659
ROC AUC (macro): 0.4228


In [14]:
if df_preds is not None and "actual_label" in df_preds.columns:
    print("Confusion Matrix (rows = actual, columns = predicted):")
    display(metrics["confusion_matrix_df"])

Confusion Matrix (rows = actual, columns = predicted):


predicted,30_days,60_days,90_days,on_time
actual,,,,
30_days,121,1253,430,226
60_days,58,307,240,146
90_days,87,160,189,730
on_time,2314,105,99,86
